# 06 — GPT-4o Investigation: Why Does GPT-4o Fool the Detector?

**Follow-up to Milestone 4** (July 2026)

The cross-generator generalisation study (notebook 05) showed that GPT-4o is the
only generator family with meaningful degradation (86.33% fake detection rate,
4.8% relative accuracy drop). Surprisingly, Janus-Pro — also autoregressive — did
**not** degrade (96.17% accuracy). This notebook investigates **why** GPT-4o is
specifically problematic through three complementary analyses:

| Analysis | Level | Question |
|----------|-------|----------|
| FFT Radial Power Spectrum | Frequency | Does GPT-4o lack the SD-specific frequency artefacts the model relies on? |
| t-SNE Feature-Space | Representation | Do GPT-4o images cluster with REAL images in the model's learned feature space? |
| Quantitative Grad-CAM | Attention | Is the model's attention measurably more diffuse on GPT-4o images? |

**Prerequisites:** Same as notebook 05 — CIFAKE zip on Drive, generalisation data downloaded.

## 0. Setup

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/ai-image-detection"
REPO_DIR = "/content/ai-image-detection"
DATA_DIR = os.path.join(REPO_DIR, "data", "raw", "cifake")
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
DRIVE_BASE = DRIVE_ROOT

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/krishi-shah/ai-image-detection.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os, zipfile, glob

os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists(os.path.join(DATA_DIR, "train")):
    print("CIFAKE already extracted — skipping.")
else:
    zip_candidates = glob.glob(os.path.join(DRIVE_ROOT, "*.zip"))
    if not zip_candidates:
        raise FileNotFoundError(
            f"No zip file found in {DRIVE_ROOT}. "
            "Upload the CIFAKE zip downloaded from Kaggle to that folder."
        )
    zip_path = zip_candidates[0]
    print(f"Found zip: {zip_path}")
    print("Extracting (this takes ~1-2 minutes)...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(DATA_DIR)
    print("Extraction complete.")

extracted = os.listdir(DATA_DIR)
if "train" not in extracted and len(extracted) == 1:
    DATA_DIR = os.path.join(DATA_DIR, extracted[0])

print(f"DATA_DIR contents: {os.listdir(DATA_DIR)}")

In [ ]:
import json
import numpy as np
import torch
from pathlib import Path

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

from src.model.detector import build_detector, load_checkpoint

CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'best_detector.pth')
if not os.path.exists(CHECKPOINT_PATH):
    CHECKPOINT_PATH = os.path.join(REPO_DIR, 'outputs/checkpoints/best_detector.pth')

model = build_detector(num_classes=2)
epoch_loaded, ckpt_metrics = load_checkpoint(model, CHECKPOINT_PATH)
model = model.to(DEVICE)
model.eval()
print(f"Model loaded from: {CHECKPOINT_PATH} (epoch {epoch_loaded})")

TEMPERATURE = 1.2189

BASELINE_PATH = os.path.join(DRIVE_BASE, 'outputs/results/baseline_results.json')
if not os.path.exists(BASELINE_PATH):
    BASELINE_PATH = os.path.join(REPO_DIR, 'outputs/results/baseline_results.json')

with open(BASELINE_PATH) as f:
    baseline_results = json.load(f)

print(f"Baseline accuracy: {baseline_results['test_accuracy']:.4f}")

## 1. Load Data

Download generalisation images (idempotent — skips if already present), then
set up loaders for all families plus CIFAKE REAL/FAKE reference sets.

In [ ]:
from scripts.download_generalisation_data import download_all_families

PER_GENERATOR = 300
GEN_DATA_DIR = os.path.join(DRIVE_BASE, 'data/generalisation')

download_results = download_all_families(
    output_dir=GEN_DATA_DIR,
    per_generator=PER_GENERATOR,
    seed=SEED,
)

from src.utils.data_loader import discover_generator_families, get_generalisation_loader

REAL_REF_DIR = os.path.join(DATA_DIR, 'test', 'REAL')
family_dirs = discover_generator_families(GEN_DATA_DIR)
family_names = [Path(d).name for d in family_dirs]
print(f"\nFamilies to analyse: {family_names}")

## 2. Analysis 1 — FFT Radial Power Spectrum

Compare the frequency signatures of images from each generator family.
If GPT-4o's power spectrum overlaps with CIFAKE REAL rather than CIFAKE FAKE,
it means GPT-4o images lack the frequency-domain artefacts the detector relies on.

In [ ]:
from PIL import Image
from src.analysis.frequency import batch_radial_spectra, plot_radial_spectra

N_FFT = 150  # images per family

def load_grayscale_images(folder, n=N_FFT):
    """Load up to n images from a folder as grayscale float arrays."""
    exts = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}
    paths = sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in exts)
    images = []
    for p in paths[:n]:
        img = Image.open(p).convert('L').resize((224, 224))
        images.append(np.array(img, dtype=np.float64) / 255.0)
    return images

# Collect spectra for each family + CIFAKE baselines
family_spectra = {}

print("Computing radial power spectra...")

# CIFAKE baselines
cifake_real = load_grayscale_images(os.path.join(DATA_DIR, 'test', 'REAL'))
family_spectra['CIFAKE_REAL'] = batch_radial_spectra(cifake_real)
print(f"  CIFAKE_REAL: {len(cifake_real)} images")

cifake_fake = load_grayscale_images(os.path.join(DATA_DIR, 'test', 'FAKE'))
family_spectra['CIFAKE_FAKE'] = batch_radial_spectra(cifake_fake)
print(f"  CIFAKE_FAKE: {len(cifake_fake)} images")

# Generator families
for fdir in family_dirs:
    name = Path(fdir).name
    fake_dir = os.path.join(fdir, 'FAKE')
    imgs = load_grayscale_images(fake_dir)
    family_spectra[name] = batch_radial_spectra(imgs)
    print(f"  {name}: {len(imgs)} images")

# Save path on Drive
FFT_SAVE_DIR = os.path.join(DRIVE_BASE, 'outputs/plots/gpt4o_investigation')
os.makedirs(FFT_SAVE_DIR, exist_ok=True)

plot_radial_spectra(
    family_spectra,
    save_path=os.path.join(FFT_SAVE_DIR, 'radial_power_spectra.png'),
    title='Radial Power Spectrum by Generator Family',
)

# Also display inline
from IPython.display import Image as IPImage, display
display(IPImage(filename=os.path.join(FFT_SAVE_DIR, 'radial_power_spectra.png')))

## 3. Analysis 2 — Feature-Space t-SNE

Extract 1536-d embeddings from the EfficientNet-B3 penultimate layer and project
to 2-D using t-SNE. If GPT-4o images cluster with CIFAKE REAL (rather than
CIFAKE FAKE), it confirms the model's internal representation treats GPT-4o
images as real-looking.

In [ ]:
from src.analysis.embeddings import extract_embeddings, compute_tsne, plot_tsne
from src.utils.data_loader import _FolderDataset, get_transforms

N_EMBED = 150  # images per family
eval_transform = get_transforms('test')

all_embeddings = []
all_family_labels = []

print("Extracting embeddings...")

# CIFAKE REAL reference
real_ds = _FolderDataset(Path(DATA_DIR) / 'test' / 'REAL', transform=eval_transform, label=1)
real_ds.paths = real_ds.paths[:N_EMBED]
real_loader = torch.utils.data.DataLoader(real_ds, batch_size=32, shuffle=False)
emb, _ = extract_embeddings(model, real_loader, DEVICE)
all_embeddings.append(emb)
all_family_labels.extend(['CIFAKE_REAL'] * len(emb))
print(f"  CIFAKE_REAL: {len(emb)} embeddings")

# CIFAKE FAKE reference
fake_ds = _FolderDataset(Path(DATA_DIR) / 'test' / 'FAKE', transform=eval_transform, label=0)
fake_ds.paths = fake_ds.paths[:N_EMBED]
fake_loader = torch.utils.data.DataLoader(fake_ds, batch_size=32, shuffle=False)
emb, _ = extract_embeddings(model, fake_loader, DEVICE)
all_embeddings.append(emb)
all_family_labels.extend(['CIFAKE_FAKE'] * len(emb))
print(f"  CIFAKE_FAKE: {len(emb)} embeddings")

# Generator families
for fdir in family_dirs:
    name = Path(fdir).name
    ds = _FolderDataset(Path(fdir) / 'FAKE', transform=eval_transform, label=0)
    ds.paths = ds.paths[:N_EMBED]
    loader = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=False)
    emb, _ = extract_embeddings(model, loader, DEVICE)
    all_embeddings.append(emb)
    all_family_labels.extend([name] * len(emb))
    print(f"  {name}: {len(emb)} embeddings")

# Concatenate and run t-SNE
combined_emb = np.concatenate(all_embeddings, axis=0)
print(f"\nTotal embeddings: {combined_emb.shape[0]} x {combined_emb.shape[1]}")
print("Running t-SNE (this may take 1-2 minutes)...")

coords_2d = compute_tsne(combined_emb, perplexity=30.0, seed=SEED)

TSNE_SAVE_DIR = os.path.join(DRIVE_BASE, 'outputs/plots/gpt4o_investigation')
plot_tsne(
    coords_2d,
    all_family_labels,
    save_path=os.path.join(TSNE_SAVE_DIR, 'feature_space_tsne.png'),
)

display(IPImage(filename=os.path.join(TSNE_SAVE_DIR, 'feature_space_tsne.png')))

## 4. Analysis 3 — Quantitative Grad-CAM Metrics

Compute entropy, peak activation ratio, and Gini coefficient on Grad-CAM
heatmaps for FAKE images from each generator. If GPT-4o shows statistically
higher entropy / lower peak ratio / lower Gini, it quantitatively confirms
the model has no focused detection signal for GPT-4o.

In [ ]:
from src.analysis.gradcam_metrics import batch_gradcam_metrics, plot_gradcam_metrics

N_GRADCAM = 100  # images per family
target_layer = model.conv_head

family_metrics = {}

print("Computing Grad-CAM metrics...")

# CIFAKE FAKE baseline
cifake_fake_ds = _FolderDataset(
    Path(DATA_DIR) / 'test' / 'FAKE', transform=eval_transform, label=0
)
cifake_fake_ds.paths = cifake_fake_ds.paths[:N_GRADCAM]
cifake_loader = torch.utils.data.DataLoader(cifake_fake_ds, batch_size=16, shuffle=False)
metrics = batch_gradcam_metrics(model, cifake_loader, target_layer, DEVICE, max_images=N_GRADCAM)
family_metrics['CIFAKE_FAKE'] = metrics
print(f"  CIFAKE_FAKE: {len(metrics)} images")

# Generator families
for fdir in family_dirs:
    name = Path(fdir).name
    ds = _FolderDataset(Path(fdir) / 'FAKE', transform=eval_transform, label=0)
    ds.paths = ds.paths[:N_GRADCAM]
    loader = torch.utils.data.DataLoader(ds, batch_size=16, shuffle=False)
    metrics = batch_gradcam_metrics(model, loader, target_layer, DEVICE, max_images=N_GRADCAM)
    family_metrics[name] = metrics
    print(f"  {name}: {len(metrics)} images")

GRADCAM_SAVE_DIR = os.path.join(DRIVE_BASE, 'outputs/plots/gpt4o_investigation')
plot_gradcam_metrics(family_metrics, save_dir=GRADCAM_SAVE_DIR)

# Display all three plots
for fname in ['gradcam_entropy_comparison.png', 'gradcam_peak_ratio_comparison.png', 'gradcam_gini_comparison.png']:
    fpath = os.path.join(GRADCAM_SAVE_DIR, fname)
    if os.path.exists(fpath):
        display(IPImage(filename=fpath))

## 5. Summary Table

In [ ]:
import pandas as pd

# Summarise Grad-CAM metrics per family
summary_rows = []
for fam, metrics_list in family_metrics.items():
    entropies = [m['entropy'] for m in metrics_list]
    peaks = [m['peak_ratio'] for m in metrics_list]
    ginis = [m['gini'] for m in metrics_list]
    summary_rows.append({
        'Family': fam,
        'Mean Entropy': f"{np.mean(entropies):.3f}",
        'Mean Peak Ratio': f"{np.mean(peaks):.3f}",
        'Mean Gini': f"{np.mean(ginis):.3f}",
        'Std Entropy': f"{np.std(entropies):.3f}",
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary.style.set_caption('Grad-CAM Attention Metrics by Generator Family'))

# Save results JSON
results_dir = os.path.join(DRIVE_BASE, 'outputs/results')
os.makedirs(results_dir, exist_ok=True)

investigation_results = {
    'gradcam_metrics': {
        fam: {
            'mean_entropy': float(np.mean([m['entropy'] for m in mlist])),
            'std_entropy': float(np.std([m['entropy'] for m in mlist])),
            'mean_peak_ratio': float(np.mean([m['peak_ratio'] for m in mlist])),
            'std_peak_ratio': float(np.std([m['peak_ratio'] for m in mlist])),
            'mean_gini': float(np.mean([m['gini'] for m in mlist])),
            'std_gini': float(np.std([m['gini'] for m in mlist])),
            'n_images': len(mlist),
        }
        for fam, mlist in family_metrics.items()
    },
}

results_path = os.path.join(results_dir, 'gpt4o_investigation.json')
with open(results_path, 'w') as f:
    json.dump(investigation_results, f, indent=2)
print(f"\nResults saved to: {results_path}")

## 6. Key Findings

### Interpretation Guide

| Analysis | GPT-4o is different | GPT-4o is similar |
|----------|--------------------|-----------|
| **FFT spectrum** | GPT-4o's spectrum matches REAL (lacks SD artefact frequencies) | Something else explains the gap |
| **Feature space (t-SNE)** | GPT-4o clusters with REAL, not FAKE | Model sees them as fake-like but still misclassifies (threshold issue) |
| **Grad-CAM metrics** | GPT-4o has higher entropy / lower Gini / lower peak ratio | Attention is similar but on wrong features |

### Conclusions

Fill in after running the analyses above:

- **FFT:** *[Does GPT-4o's frequency profile overlap with REAL or FAKE?]*
- **t-SNE:** *[Where does GPT-4o cluster — with REAL, FAKE, or separately?]*
- **Grad-CAM:** *[Is GPT-4o entropy significantly higher than other families?]*
- **Overall:** *[Does the evidence support a frequency-level, representation-level, or attention-level explanation?]*